# Phase 5 Slice 5A notebook demo

This notebook uses the real ADK API server session and run endpoints against the local Solo gateway. It keeps the synthetic JWT trust chain private and reads the Token B / Token C values from a local file-backed provider.

In [ ]:
import json
import os
import uuid

import requests

from agents.token_provider import get_token_provider

base_url = os.getenv("ADK_API_BASE", "http://127.0.0.1:18080")
flow_id = uuid.uuid4().hex[:12]
provider = get_token_provider()
print({
    "api_url": base_url,
    "flow_id": flow_id,
    "token_a": provider.token_a[:24] + '...',
    "token_b": provider.token_b[:24] + '...',
    "token_c": provider.token_c[:24] + '...',
})

In [ ]:
def create_session(app_name: str, user_id: str = "user-12345"):
    url = f"{base_url}/apps/{app_name}/users/{user_id}/sessions"
    response = requests.post(
        url,
        json={"session_name": f"phase5-{uuid.uuid4().hex[:8]}"},
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {provider.token_a}",
            "X-Flow-ID": flow_id,
        },
        timeout=30,
    )
    response.raise_for_status()
    return response.json()


def run_prompt(app_name: str, user_id: str, session_id: str, prompt: str):
    url = f"{base_url}/apps/{app_name}/users/{user_id}/sessions/{session_id}/run"
    response = requests.post(
        url,
        json={"input": {"content": prompt}},
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {provider.token_a}",
            "X-Flow-ID": flow_id,
        },
        timeout=60,
    )
    response.raise_for_status()
    return response.json()

session = create_session("agent_a")
result = run_prompt("agent_a", "user-12345", session["id"], "delegate to agent b: what is the balance for acct-1001?")
print(json.dumps(result, indent=2)[:2000])